# Предобработка и признаки

Сырой `DataFrame` → инженерия признаков → **стратифицированный train/test** → кодирование категорий, при этом **`county_encoded` строится только по train** (без утечки тестовых уездов; редкие уезды в test → код как у `unknown`).

Импутация и масштабирование — fit только на train.

Артефакт `data/processed/ml_ready.joblib` используется в ноутбуках 04 и 05.

**Colab:** [colab_quickstart.ipynb](colab_quickstart.ipynb); при необходимости `%cd /content/water-quality-ee`.

**Для новичков:** подробный разбор — [`docs/ml_metrics_guide.md`](../docs/ml_metrics_guide.md).


## Зачем нужна инженерия признаков, если нормативы и так известны?

Официальный вердикт Terviseamet — это детерминированное пороговое правило:
хоть один параметр вышел за норму → `compliant = 0`.

**Мы не можем применить то же правило напрямую** по двум причинам:

**1. Пропуски — главная проблема.** В типичной пробе измерено 2–4 параметра из 15.
Когда нитраты не измерялись, мы не знаем, нарушают ли они норму. Правило говорит "нет данных".
Модель учится отвечать по частичным данным — используя то, что измерено, и сам факт отсутствия измерений.

**2. Взаимодействие параметров.** Железо 0.18 мг/л + мутность 3.9 NTU — каждый ниже нормы,
но их совместное присутствие "на грани" исторически коррелирует с нарушениями по цветности или марганцу.

### Что делает инженерия признаков в этом ноутбуке

| Тип признака | Пример | Зачем |
|---|---|---|
| Сырые значения | `iron = 0.18` | Базовый сигнал |
| Отношение к норме | `iron_ratio = 0.18 / 0.2 = 0.90` | Нормализует к масштабу нарушения |
| Индикатор пропуска | `color_missing = 1` | Факт неизмерения несёт информацию о типе пробы |
| Сезонные признаки | `is_summer = 1`, `season_summer = 1` | Летом в 4× больше нарушений в открытых водоёмах |
| Тип домена | `domain_veevark = 1` | Водопровод vs купание — разный профиль нарушений |

`iron_ratio > 1` означает нарушение конкретно по этому параметру. Но модель учится видеть
вероятность нарушения **пробы в целом** по комбинации всех доступных сигналов.

In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

def _project_root() -> Path:
    spec = importlib.util.find_spec('data_loader')
    if spec is not None and getattr(spec, 'origin', None):
        cand = Path(spec.origin).resolve().parent.parent
        if (cand / 'data' / 'raw').is_dir() or (cand / 'src' / 'data_loader.py').is_file():
            return cand
    env = os.environ.get('WATER_QUALITY_EE_ROOT', '').strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / 'src' / 'data_loader.py').is_file():
            return p
    cwd = Path.cwd().resolve()
    try:
        r = subprocess.run(
            ['git', 'rev-parse', '--show-toplevel'],
            cwd=cwd, capture_output=True, text=True, timeout=15,
        )
        if r.returncode == 0 and r.stdout.strip():
            p = Path(r.stdout.strip()).resolve()
            if (p / 'src' / 'data_loader.py').is_file():
                return p
    except (FileNotFoundError, OSError, subprocess.TimeoutExpired):
        pass
    for root in [cwd, *list(cwd.parents)[:28]]:
        if (root / 'src' / 'data_loader.py').is_file():
            return root
    raise RuntimeError('См. 01: pip install -e . или корректный WATER_QUALITY_EE_ROOT.')

if importlib.util.find_spec('data_loader') is None:
    sys.path.insert(0, str(_project_root() / 'src'))

import data_loader
ROOT = Path(data_loader.__file__).resolve().parent.parent

import joblib

import pandas as pd
from sklearn.model_selection import train_test_split

from data_loader import load_all
from features import (
    FEATURE_COLS,
    engineer_features,
    encode_categoricals,
    fit_county_mapping,
    impute_and_scale,
)

## 0b. Важно: дедупликация мест — `location_key`

**Это нужно знать перед любой агрегацией по месту.**

Terviseamet переименовывает места в XML между годами (например, `'Harku järve supluskoht'` в 2021 и
`'Harku järve rand'` в 2025 — одно место). Без нормализации любой `groupby("location")`
даёт ложные дубли: место выглядит как два объекта.

`data_loader.load_domain()` / `load_all()` автоматически добавляет столбец `location_key` —
нормализованное имя (нижний регистр, без суффиксов типа объекта):

```python
from data_loader import normalize_location
normalize_location('Harku järve supluskoht')  # → 'harku järve'
normalize_location('Harku järve rand')        # → 'harku järve'  ← то же место
```

**Правило:** при подсчёте уникальных мест и при group-by по месту — используйте `location_key`.

> Подробный анализ дублей — ноутбук **02** (раздел 1b).

## 1. Загрузка, признаки, split и кодирование уезда

### Пояснение для начинающих
- `load_all` — сырой `DataFrame`; `engineer_features` — время, отношения к нормам, индикаторы пропусков (**ещё без** one-hot домена/сезона и без `county_encoded`).
- Сначала **`train_test_split`** по индексу (стратификация по `compliant`), затем `fit_county_mapping` **только на train**, затем `encode_categoricals` на всей таблице — так тест не задаёт словарь уездов.

### Результат
Матрицы `X_train`, `X_test`, `y_train`, `y_test` и число уровней уезда в train.

In [2]:
raw = load_all(use_cache=True)
df_eng = engineer_features(raw)
train_ix, test_ix = train_test_split(
    df_eng.index,
    test_size=0.2,
    random_state=42,
    stratify=df_eng["compliant"],
)
county_mapping = fit_county_mapping(df_eng.loc[train_ix, "county"])
df_enc = encode_categoricals(df_eng, county_mapping=county_mapping)
y = df_enc["compliant"].astype(int)
available = [c for c in FEATURE_COLS if c in df_enc.columns]
X = df_enc[available]
X_train, X_test = X.loc[train_ix], X.loc[test_ix]
y_train, y_test = y.loc[train_ix], y.loc[test_ix]
print(f"Признаков: {X.shape[1]}, уникальных уездов в train (с unknown): {len(county_mapping)}")
X.shape, y.shape, len(X_train), len(X_test)

[data_loader] Скачиваю supluskoha за 2026…


[data_loader] Год 2026 недоступен: 404 Client Error: Not Found for url: https://vtiav.sm.ee/index.php/opendata/supluskoha_veeproovid_2026.xml
[data_loader] Кэш: supluskoha_2025.xml
[data_loader] Кэш: supluskoha_2024.xml
[data_loader] Кэш: supluskoha_2023.xml
[data_loader] Кэш: supluskoha_2022.xml
[data_loader] Кэш: supluskoha_2021.xml
[data_loader] supluskoha: 4031 проб, 4031 с известным статусом
[data_loader] Кэш: veevark_2026.xml
[data_loader] Кэш: veevark_2025.xml
[data_loader] Кэш: veevark_2024.xml


[data_loader] Кэш: veevark_2023.xml
[data_loader] Кэш: veevark_2022.xml
[data_loader] Кэш: veevark_2021.xml


[data_loader] veevark: 34563 проб, 34563 с известным статусом
[data_loader] Кэш: basseinid_2026.xml
[data_loader] Кэш: basseinid_2025.xml
[data_loader] Кэш: basseinid_2024.xml
[data_loader] Кэш: basseinid_2023.xml
[data_loader] Кэш: basseinid_2022.xml
[data_loader] Кэш: basseinid_2021.xml


[data_loader] basseinid: 30477 проб, 30477 с известным статусом
[data_loader] Кэш: joogivesi_2026.xml
[data_loader] Кэш: joogivesi_2025.xml
[data_loader] Кэш: joogivesi_2024.xml
[data_loader] Кэш: joogivesi_2023.xml
[data_loader] Кэш: joogivesi_2022.xml
[data_loader] Кэш: joogivesi_2021.xml
[data_loader] joogivesi: 376 проб, 376 с известным статусом
[data_loader] Итого: 69447 проб из 4 доменов


[county_infer] Заполнено county не из XML: 952 строк; распределение county_source:
county_source
unknown     68495
override      952
[features] После удаления без compliant: 69447 строк
Признаков: 71, уникальных уездов в train (с unknown): 7


((69447, 71), (69447,), 55557, 13890)

## 2. Разбиение уже сделано

Train/test и `county_mapping` заданы в ячейке выше (`random_state=42`, `stratify` по `compliant`).

In [3]:
# Проверка баланса класса 0 на test
y_train.value_counts(normalize=True).round(3), y_test.value_counts(normalize=True).round(3)

(compliant
 1    0.88
 0    0.12
 Name: proportion, dtype: float64,
 compliant
 1    0.88
 0    0.12
 Name: proportion, dtype: float64)

## 3. Импутация (median) и StandardScaler — fit на train

### Пояснение для начинающих
- **Импутация медианой** заполняет пропуски типичным «серединным» значением; медиана устойчивее среднего к выбросам.
- **StandardScaler** вычитает среднее и делит на стандартное отклонение — признаки становятся сопоставимы по масштабу.
- Критично: **обучаем** (fit) преобразования **только на train**, затем **применяем** к train и test — иначе **утечка** информации из теста.

### Результат раздела (инсайт)
Таблицы `X_train_s`, `X_test_s` без NaN и с единым масштабом — готовый вход для моделей.

In [4]:
X_train_s, X_test_s = impute_and_scale(X_train, X_test)
X_train_s.head()

,e_coli,enterococci,ph,transparency,nitrates,nitrites,ammonium,fluoride,manganese,iron,...,is_summer,county_encoded,domain_supluskoha,domain_veevark,domain_basseinid,domain_joogivesi,season_summer,season_winter,season_spring,season_autumn
0,-0.014883,-0.045806,-1.131583,0.0,6.389058,-0.034637,-0.070299,-0.026391,-0.008172,-0.045127,...,-0.566978,0.10937,-0.249265,-0.995134,1.131706,-0.074298,-0.566978,1.796955,-0.571745,-0.614209
1,-0.014883,-0.045806,-0.006827,0.0,-0.173186,-0.034637,-0.070299,-0.026391,-0.008172,-0.045127,...,-0.566978,0.10937,-0.249265,-0.995134,1.131706,-0.074298,-0.566978,1.796955,-0.571745,-0.614209
2,-0.014883,-0.045806,-0.006827,0.0,-0.173186,-0.034637,-0.070299,-0.026391,-0.008172,-0.045127,...,1.763736,0.10937,-0.249265,-0.995134,1.131706,-0.074298,1.763736,-0.556497,-0.571745,-0.614209
3,-0.014883,-0.045806,-0.006827,0.0,-0.173186,-0.034637,-0.070299,-0.026391,-0.008172,-0.045127,...,1.763736,0.10937,-0.249265,1.004890,-0.883622,-0.074298,1.763736,-0.556497,-0.571745,-0.614209
4,0.050805,0.503111,-0.006827,0.0,-0.173186,-0.034637,-0.070299,-0.026391,-0.008172,-0.045127,...,1.763736,0.10937,4.011790,-0.995134,-0.883622,-0.074298,1.763736,-0.556497,-0.571745,-0.614209


## 4. Сохранение для моделей (ноутбуки 04–05)

### Пояснение для начинающих
- `joblib.dump` сохраняет словарь с матрицами и именами признаков в **один бинарный файл**.
- Следующие ноутбуки **не пересчитывают** split и предобработку — берут готовое, чтобы метрики были сопоставимы.

### Результат раздела (инсайт)
Файл `data/processed/ml_ready.joblib` — единый «чекпоинт» пайплайна после EDA.

In [ ]:
out_dir = ROOT / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)
bundle = {
    'X_train': X_train_s,
    'X_test': X_test_s,
    'y_train': y_train,
    'y_test': y_test,
    'feature_names': list(X_train_s.columns),
    'county_mapping': county_mapping,
}
path = out_dir / 'ml_ready.joblib'
joblib.dump(bundle, path)
print('Сохранено:', path)

Сохранено: /workspace/data/processed/ml_ready.joblib
